# Análise de Receita

## Objetivo

Analisar a evolução da receita ao longo do tempo.

---

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [54]:
# ============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ============================================================================

# Descobre automaticamente a raiz do projeto
# Usando as funçõs OS pra garantir compatibilidade entre sistemas operacionais

# Pega o arquivo atual
from pathlib import Path
BASE_PATH = Path.cwd()

# Sobe na hierarquia de pastas até encontrar a raiz do projeto
while BASE_PATH.name != "PROJETO-INTEGRADOR--GRUPO-37" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

# Caminhos dos dados
RAW_DATA_PATH = BASE_PATH / "data" / "raw"
CLEANED_DATA_PATH = BASE_PATH / "data" / "cleaned" / "pessoa_6"
FIGURES_PATH = BASE_PATH / "reports" / "pessoa_6" / "figures"

print("=" * 80)
print("ETAPA 1: CONFERÊNCIA DE CAMINHOS")
print("=" * 80, "\n")
# Mostrar os caminhos (para verificar)
print(f"─ Raiz do projeto: {BASE_PATH}")
print(f"─ Dados brutos: {RAW_DATA_PATH}")
print(f"─ Dados limpos: {CLEANED_DATA_PATH}")
print(f"─ Dados das imagens geradas: {FIGURES_PATH}", "\n")

# ============================================================================
# CONFIGURAÇÃO DA VISUALIZAÇÃO DE DADOS
# ============================================================================

# Mostrar todas as colunas, sem o corte no meio pra caber no terminal.
pd.set_option('display.max_columns', None)
# Largura automática.
pd.set_option('display.width', None)
# Limite de caracteres por célula, padronizar a visualização.
pd.set_option('display.max_colwidth', 30)

ETAPA 1: CONFERÊNCIA DE CAMINHOS

─ Raiz do projeto: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37
─ Dados brutos: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37\data\raw
─ Dados limpos: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37\data\cleaned\pessoa_6
─ Dados das imagens geradas: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37\reports\pessoa_6\figures 



## Carregamento dos Dados Tratados

---

In [3]:
# ============================================================================
# CARREGAMENTO DOS DADOS
# ============================================================================

# Carregando a tabela de pagamentos
df_payments = pd.read_csv(os.path.join(CLEANED_DATA_PATH, "payments_clean.csv"))

# Carregando a tabela de itens de pedido
df_order_items = pd.read_csv(os.path.join(CLEANED_DATA_PATH, "order_items_clean.csv"))

# Carregando a tabela de pedidos
df_orders = pd.read_csv(os.path.join(CLEANED_DATA_PATH, "orders_clean.csv"))

## Converter Datetime dos Dados

 Os dados precisam ser convertidos novamente. Quando salvamos em CSV eles voltam pra object.
 
---

In [4]:
# ============================================================================
# CONVERTENDO DADOS
# ============================================================================

# Ajustando o tipo das colunas de data:
df_order_items['shipping_limit_date'] = pd.to_datetime(df_order_items['shipping_limit_date'], errors='coerce')
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'], errors='coerce')
df_orders['order_approved_at'] = pd.to_datetime(df_orders['order_approved_at'], errors='coerce')
df_orders['order_delivered_carrier_date'] = pd.to_datetime(df_orders['order_delivered_carrier_date'], errors='coerce')
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'], errors='coerce')
df_orders['order_estimated_delivery_date'] = pd.to_datetime(df_orders['order_estimated_delivery_date'], errors='coerce')

---

## Conferência das Tratativas

In [9]:
# ============================================================================
# CONFERÊNCIA DOS DADOS TRATADOS
# ============================================================================

print("\n" , "Colunas e tipos de dados - df_orders:")
print("-" * 80)
print(df_orders.dtypes, "\n")

print("\n" , "Colunas e tipos de dados - df_order_items:")
print("-" * 80)
print(df_order_items.dtypes, "\n")


 Colunas e tipos de dados - df_orders:
--------------------------------------------------------------------------------
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object 


 Colunas e tipos de dados - df_order_items:
--------------------------------------------------------------------------------
order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object 



---

## Preparação dos Dados

In [26]:
# ============================================================================
# PREPARAÇÃO DOS DADOS PARA ANÁLISE
# ============================================================================

# Vamos começar com os dados de evolução de receita. Left join simples.
df_receita = df_payments.merge(
    df_orders[['order_id', 'order_purchase_timestamp', 'order_status']],
    on='order_id',
    how='left'
)

print(f'Primeiras 3 linhas:')
display(df_receita[['order_id', 'payment_value', 'order_purchase_timestamp', 'order_status']].head(3))

Primeiras 3 linhas:


,order_id,payment_value,order_purchase_timestamp,order_status
0,b81ef226f3fe1789b1e8b2acac...,99.33,2018-04-25 22:01:49,delivered
1,a9810da82917af2d9aefd1278f...,24.39,2018-06-26 11:01:38,delivered
2,25e8ea4e93396b6fa0d3dd708e...,65.71,2017-12-12 11:19:55,delivered


In [13]:
# ============================================================================
# PREPARAÇÃO DOS DADOS PARA ANÁLISE
# ============================================================================

# Trazendo primeiro apenas os que foram entregues.
df_receita_entregue = df_receita[df_receita ['order_status'] == 'delivered'].copy()

# Outras métricas relevantes.
df_cancelados = df_receita[df_receita ['order_status'] == 'canceled'].copy()
df_processando = df_receita[df_receita ['order_status'] == 'processing'].copy()
df_pendentes = df_receita[df_receita ['order_status'] == 'pending'].copy()
df_indisponiveis = df_receita[df_receita ['order_status'] == 'unavailable'].copy()

print(f"Total de registros: {len(df_receita):,}")
print(f"Pedidos entregues: {len(df_receita_entregue):,}")
print(f"Percentual entregue: {(len(df_receita_entregue) / len(df_receita) * 100):.2f}%")
print(f"Pedidos cancelados: {len(df_cancelados):,}")
print(f"Percentual cancelados: {(len(df_cancelados) / len(df_receita) * 100):.2f}%")
print(f"Pedidos em processamento: {len(df_processando):,}")
print(f"Percentual em processamento: {(len(df_processando) / len(df_receita) * 100):.2f}%")
print(f"Pedidos pendentes: {len(df_pendentes):,}")
print(f"Percentual pendentes: {(len(df_pendentes) / len(df_receita) * 100):.2f}%")
print(f"Pedidos indisponíveis: {len(df_indisponiveis):,}")
print(f"Percentual indisponíveis: {(len(df_indisponiveis) / len(df_receita) * 100):.2f}%")

Total de registros: 103,886
Pedidos entregues: 100,756
Percentual entregue: 96.99%
Pedidos cancelados: 664
Percentual cancelados: 0.64%
Pedidos em processamento: 319
Percentual em processamento: 0.31%
Pedidos pendentes: 0
Percentual pendentes: 0.00%
Pedidos indisponíveis: 649
Percentual indisponíveis: 0.62%


In [25]:
# ============================================================================
# PREPARAÇÃO DOS DADOS PARA ANÁLISE
# ============================================================================

# Iniciar as tratativas de time intelligence
df_receita_entregue['ano'] = df_receita_entregue['order_purchase_timestamp'].dt.year
df_receita_entregue['mes'] = df_receita_entregue['order_purchase_timestamp'].dt.month
df_receita_entregue['ano_mes'] = df_receita_entregue['order_purchase_timestamp'].dt.to_period('M')

print(f'Primeiras 3 linhas:')
display(df_receita_entregue[['order_purchase_timestamp', 'ano', 'mes', 'ano_mes', 'payment_value']].head(3))

Primeiras 3 linhas:


,order_purchase_timestamp,ano,mes,ano_mes,payment_value
0,2018-04-25 22:01:49,2018,4,2018-04,99.33
1,2018-06-26 11:01:38,2018,6,2018-06,24.39
2,2017-12-12 11:19:55,2017,12,2017-12,65.71


In [47]:
# ============================================================================
# PREPARAÇÃO DOS DADOS PARA ANÁLISE
# ============================================================================

# Soma da receita mensal
df_receita_mensal = df_receita_entregue.groupby('ano_mes').agg({
    'payment_value': 'sum',
    'order_id': 'nunique'
}).reset_index()

df_receita_mensal.columns = ['ano_mes', 'receita_total', 'total_pedidos']
df_receita_mensal['ano_mes'] = df_receita_mensal['ano_mes'].astype(str)

# Adicionar nova coluna
df_receita_mensal['ticket_medio'] = df_receita_mensal['receita_total'] / df_receita_mensal['total_pedidos']

print(f'Primeiras 3 linhas:')
display(df_receita_mensal.head(3))

Primeiras 3 linhas:


,ano_mes,receita_total,total_pedidos,ticket_medio
0,2016-10,46566.71,265,175.723434
1,2016-12,19.62,1,19.620000
2,2017-01,127545.67,750,170.060893


In [48]:
# ============================================================================
# ESTATÍSTICAS RELEVANTES
# ============================================================================

# Como ainda não tenho o escopo definido dos dados que serão mostrados
# vou trazer algumas estatísticas que podem ser relevantes
print(f"Receita Total (período completo): R$ {df_receita_mensal['receita_total'].sum():,.2f}")
print(f"Receita Média Mensal: R$ {df_receita_mensal['receita_total'].mean():,.2f}")
print(f"Total de Pedidos: {df_receita_mensal['total_pedidos'].sum():,}")
print(f"Ticket Médio Geral: R$ {df_receita_mensal['ticket_medio'].mean():.2f}")
print()

# Melhor e pior mês
melhor_mes = df_receita_mensal.loc[df_receita_mensal['receita_total'].idxmax()]
pior_mes = df_receita_mensal.loc[df_receita_mensal['receita_total'].idxmin()]

print("MELHOR MÊS:")
print(f"- Período: {melhor_mes['ano_mes']}")
print(f"- Receita: R$ {melhor_mes['receita_total']:,.2f}")
print(f"- Pedidos: {melhor_mes['total_pedidos']:,}")
print()

print("PIOR MÊS:")
print(f"- Período: {pior_mes['ano_mes']}")
print(f"- Receita: R$ {pior_mes['receita_total']:,.2f}")
print(f"- Pedidos: {pior_mes['total_pedidos']:,}")
print()

Receita Total (período completo): R$ 15,422,461.77
Receita Média Mensal: R$ 701,020.99
Total de Pedidos: 96,477
Ticket Médio Geral: R$ 154.93

MELHOR MÊS:
- Período: 2017-11
- Receita: R$ 1,153,528.05
- Pedidos: 7,289

PIOR MÊS:
- Período: 2016-12
- Receita: R$ 19.62
- Pedidos: 1



---

## Gráfico 1: Evolução da Receita Mensal

In [67]:
# ============================================================================
# GRÁFICO RECEITA MENSAL - EVOLUÇÃO
# ============================================================================

# Criar figura
fig = go.Figure()

# Adicionar linha de receita
fig.add_trace(go.Scatter(
    x=df_receita_mensal['ano_mes'],
    y=df_receita_mensal['receita_total'],
    mode='lines+markers',
    name='Receita Mensal',
    line=dict(color='#1f77b4', width=3),     
    marker=dict(size=8),                      # Bolinhas dos marcadores
    hovertemplate='<b>%{x}</b><br>Receita: R$ %{y:,.2f}<extra></extra>'    # Aqui ele define qual o dado vai ser mostrado durante o hover
))

# Configurar layout
fig.update_layout(
    title={
        'text': '<b>Evolução da Receita Mensal</b>',
        'x': 0.5,                    # Centralizar
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    xaxis_title='Mês/Ano',
    yaxis_title='Receita (R$)',
    template='plotly_white',         # Tema branco limpo
    height=500,
    hovermode='x unified'            # Hover unificado
)

# Mostrar gráfico
fig.show()

# Salvar o gráfico como HTML
caminho_grafico = FIGURES_PATH / "receita_mensal_evolucao.html"
fig.write_html(caminho_grafico)

print(f"Gráfico salvo em: {caminho_grafico}")
print()

Gráfico salvo em: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37\reports\pessoa_6\figures\receita_mensal_evolucao.html



---

## Gráfico 2: Evolução do Ticket Médio

In [ ]:
# ============================================================================
# GRÁFICO TICKET MÉDIO - EVOLUÇÃO
# ============================================================================

display(df_receita_mensal[['ano_mes', 'receita_total', 'total_pedidos', 'ticket_medio']])
print(f"Ticket Médio Geral: R$ {df_receita_mensal['ticket_medio'].mean():.2f}")
print(f"Maior Ticket Médio: R$ {df_receita_mensal['ticket_medio'].max():.2f} ({df_receita_mensal.loc[df_receita_mensal['ticket_medio'].idxmax(), 'ano_mes']})")
print(f"Menor Ticket Médio: R$ {df_receita_mensal['ticket_medio'].min():.2f} ({df_receita_mensal.loc[df_receita_mensal['ticket_medio'].idxmin(), 'ano_mes']})", '\n')

# ============================================================================
# GRÁFICO RECEITA MENSAL - EVOLUÇÃO
# ============================================================================

# Criar figura
fig = go.Figure()

# Adicionar linha de receita
fig.add_trace(go.Scatter(
    x=df_receita_mensal['ano_mes'],
    y=df_receita_mensal['ticket_medio'],
    mode='lines+markers',
    name='Ticket Médio',
    line=dict(color='#9b59b6', width=3),     
    marker=dict(size=8),                      # Bolinhas dos marcadores
    fill='tozeroy',                           # Preencher até o zero
    fillcolor='rgba(155, 89, 182, 0.2)',      # Fill do gráfico, igual gráfico de área
    
    # Com base na documentação, consigo trazer aqui mais informações pra colocar dentro do hover
    customdata=df_receita_mensal[['total_pedidos', 'receita_total']],
    
    hovertemplate=(
        '<b>%{x}</b><br>' +
        'Ticket médio: R$ %{y:,.2f}</br>' +
        'Pedidos: %{customdata[0]:,}<br>' +   # Chama a variável com base no índice
        'Receita Total: R$ %{customdata[1]:,.2f}' +  # Chama a variável com base no índice
        '<extra></extra>'    
)))

# Adicionar linha de média geral (referência)
media_geral = df_receita_mensal['ticket_medio'].mean()
fig.add_hline(
    y=media_geral,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Média: R$ {media_geral:.2f}",
    annotation_position="right"
)

# Configurar layout
fig.update_layout(
    title={
        'text': '<b>Evolução do Ticket Médio Mensal</b>',
        'x': 0.5,                    # Centralizar
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    xaxis_title='Mês/Ano',
    yaxis_title='Ticket Médio (R$)',
    template='plotly_white',         # Tema branco limpo
    height=500,
    hovermode='x unified'            # Hover unificado
)

# Mostrar gráfico
fig.show()

# Salvar o gráfico como HTML
caminho_grafico = FIGURES_PATH / "ticket_medio_evolucao.html"
fig.write_html(caminho_grafico)

print(f"Gráfico salvo em: {caminho_grafico}")
print()

,ano_mes,receita_total,total_pedidos,ticket_medio
0,2016-10,46566.71,265,175.723434
1,2016-12,19.62,1,19.620000
2,2017-01,127545.67,750,170.060893
3,2017-02,271298.65,1653,164.125015
4,2017-03,414369.39,2546,162.753099
5,2017-04,390952.18,2303,169.757785
6,2017-05,567066.73,3546,159.917296
7,2017-06,490225.60,3135,156.371802
8,2017-07,566403.93,3872,146.282007
9,2017-08,646000.61,4193,154.066446


🎯 Ticket Médio Geral: R$ 154.93
📈 Maior Ticket Médio: R$ 175.72 (2016-10)
📉 Menor Ticket Médio: R$ 19.62 (2016-12) 



Gráfico salvo em: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37\reports\pessoa_6\figures\ticket_medio_evolucao.html



---

## Gráfico 3: Sazonalidade

In [91]:
# ============================================================================
# GRÁFICO DE SAZONALIDADE - ACUMULADO MES A MES
# ============================================================================

# Extrair mês (1-12) da data de compra
df_receita_entregue['mes_numero'] = df_receita_entregue['order_purchase_timestamp'].dt.month

df_sazonalidade = df_receita_entregue.groupby('mes_numero').agg({
    'payment_value': 'sum',
    'order_id': 'nunique'
    }).reset_index()
# Renomear colunas
df_sazonalidade.columns = ['mes_numero', 'receita_total', 'total_pedidos']

# Calcular ticket médio
df_sazonalidade['ticket_medio'] = df_sazonalidade['receita_total'] / df_sazonalidade['total_pedidos']

# Adicionar nome do mês (número para text)
meses_nomes = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}
df_sazonalidade['mes_nome'] = df_sazonalidade['mes_numero'].map(meses_nomes)

# Reordenar colunas para melhor visualização
df_sazonalidade = df_sazonalidade[['mes_numero', 'mes_nome', 'receita_total', 'total_pedidos', 'ticket_medio']]

display(df_sazonalidade)

# Identificar melhor e pior mês
melhor_mes = df_sazonalidade.loc[df_sazonalidade['receita_total'].idxmax()]
pior_mes = df_sazonalidade.loc[df_sazonalidade['receita_total'].idxmin()]

print()
print(f"Melhor Mês: {melhor_mes['mes_nome']}")
print(f"Receita Total: R$ {melhor_mes['receita_total']:,.2f}")
print(f"Pedidos: {melhor_mes['total_pedidos']:,}")
print(f"Ticket Médio: R$ {melhor_mes['ticket_medio']:.2f}")
print()
print(f"Pior Mês: {pior_mes['mes_nome']}")
print(f"Receita Total: R$ {pior_mes['receita_total']:,.2f}")
print(f"Pedidos: {pior_mes['total_pedidos']:,}")
print(f"Ticket Médio: R$ {pior_mes['ticket_medio']:.2f}")
print()

# Criar figura
fig = go.Figure()

# Agora quero testar um gráfico de barras
fig.add_trace(go.Bar(
    x=df_sazonalidade['mes_nome'],
    y=df_sazonalidade['receita_total'],
    marker_color='#3498db',  # Azul
    text=df_sazonalidade['receita_total'],
    texttemplate='R$ %{text:,.0f}',
    textposition='outside',
# Tooltip personalizado novamente, acho que agrega bastante
    customdata=df_sazonalidade[['total_pedidos', 'ticket_medio']],
    hovertemplate=(
        '<b>%{x}</b><br>' +
        'Receita Total: R$ %{y:,.2f}<br>' +
        'Pedidos: %{customdata[0]:,}<br>' +
        'Ticket Médio: R$ %{customdata[1]:.2f}' +
        '<extra></extra>'
    )
))

# Configurar layout
fig.update_layout(
    title={
        'text': '<b>Sazonalidade - Receita por Mês do Ano</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    xaxis_title='Mês',
    yaxis_title='Receita Total (R$)',
    template='plotly_white',
    height=500,
    showlegend=False,
    
    # Se não colocar isso, o hover esta ficando azul, ainda não encontrei o motivo
    hoverlabel=dict(
        bgcolor="white",        # Fundo branco
        font_size=14,           # Tamanho da fonte
        font_family="Arial",    # Fonte
        font_color="black"      # Texto preto
    )
)

# Mostrar
fig.show()

# Salvar o gráfico como HTML
caminho_grafico = FIGURES_PATH / "sazonalidade.html"
fig.write_html(caminho_grafico)

print(f"Gráfico salvo em: {caminho_grafico}")
print()

,mes_numero,mes_nome,receita_total,total_pedidos,ticket_medio
0,1,Janeiro,1206152.53,7819,154.259180
1,2,Fevereiro,1237809.53,8208,150.805255
2,3,Março,1535047.39,9549,160.754780
3,4,Abril,1523886.13,9101,167.441614
4,5,Maio,1695903.42,10295,164.730784
5,6,Junho,1502316.28,9234,162.693987
6,7,Julho,1594307.79,10031,158.938071
7,8,Agosto,1631414.89,10544,154.724477
8,9,Setembro,701169.99,4150,168.956624
9,10,Outubro,797706.98,4743,168.186165



Melhor Mês: Maio
Receita Total: R$ 1,695,903.42
Pedidos: 10,295
Ticket Médio: R$ 164.73

Pior Mês: Setembro
Receita Total: R$ 701,169.99
Pedidos: 4,150
Ticket Médio: R$ 168.96



Gráfico salvo em: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37\reports\pessoa_6\figures\sazonalidade.html



---

## Gráfico 4: Parcelas x Valor Médio

In [90]:
# ============================================================================
# GRÁFICO DE PARCELAS X VALOR MEDIO - SCATTER
# ============================================================================

# Agrupar por número de parcelas
df_parcelamento = df_receita_entregue.groupby('payment_installments').agg({
    'payment_value': ['sum', 'mean', 'count'],
    'order_id': 'nunique'
}).reset_index()

# Simplificar nomes das colunas
df_parcelamento.columns = ['num_parcelas', 'receita_total', 'valor_medio', 'num_transacoes', 'num_pedidos']

# Calcular percentual de pedidos
df_parcelamento['percentual_pedidos'] = (df_parcelamento['num_pedidos'] / df_parcelamento['num_pedidos'].sum()) * 100

# Ordenar por número de parcelas
df_parcelamento = df_parcelamento.sort_values('num_parcelas')

# Criar figura
fig = go.Figure()

# Adicionar scatter plot
fig.add_trace(go.Scatter(
    x=df_parcelamento['num_parcelas'],
    y=df_parcelamento['valor_medio'],
    mode='markers',  # Apenas marcadores (sem linha)
    marker=dict(
        size=df_parcelamento['num_pedidos'],  # Tamanho da bolha = nº de pedidos
        sizemode='area',
        sizeref=2.*max(df_parcelamento['num_pedidos'])/(40.**2),  # Escala do tamanho
        color=df_parcelamento['num_pedidos'],  # Cor baseada no nº de pedidos
        colorscale='Blues',  # Escala de azul
        showscale=True,  # Mostrar barra de cores
        colorbar=dict(title="Nº de<br>Pedidos"),
        line=dict(width=1, color='white')  # Borda branca nas bolhas
    ),
    
    # Tooltip personalizado
    customdata=df_parcelamento[['num_pedidos', 'receita_total', 'percentual_pedidos']],
    hovertemplate=(
        '<b>%{x}x Parcelas</b><br>' +
        'Valor Médio: R$ %{y:,.2f}<br>' +
        'Pedidos: %{customdata[0]:,} (%{customdata[2]:.1f}%)<br>' +
        'Receita Total: R$ %{customdata[1]:,.2f}' +
        '<extra></extra>'
    ),
    name=''
))

# Configurar layout
fig.update_layout(
    title={
        'text': '<b>Relação: Número de Parcelas x Valor Médio da Compra</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#2c3e50'}
    },
    xaxis_title='Número de Parcelas',
    yaxis_title='Valor Médio da Compra (R$)',
    template='plotly_white',
    height=600,
    showlegend=False,
    
    # Customizar tooltip
    hoverlabel=dict(
        bgcolor="white",
        font_size=14,
        font_color="black",
        bordercolor="#cccccc"
    )
)

# Ajustar eixo X para mostrar apenas números inteiros
fig.update_xaxes(dtick=1)  # Incremento de 1 em 1

# Mostrar
fig.show()


# Salvar o gráfico como HTML
caminho_grafico = FIGURES_PATH / "parcela_versus_valor_medio.html"
fig.write_html(caminho_grafico)

print(f"Gráfico salvo em: {caminho_grafico}")
print()

Gráfico salvo em: c:\Users\lucas\Documents\Projetos\SENAC - Projeto Integrador\PROJETO-INTEGRADOR--GRUPO-37\reports\pessoa_6\figures\parcela_versus_valor_medio.html

